# 用 Claude Managed Agents 构建 Slack 数据分析机器人

## 引言

你将把 [﷐0﷑](data_analyst_agent.ipynb) 中的智能体封装进一个 Slack 机器人，该机器人使用 [Bolt for Python](https://docs.slack.dev/tools/bolt-python/) 构建，这是 Slack 官方的应用开发框架。在消息中提及机器人并附上 CSV 附件，它就会在同一会话串中回发一份叙述式报告。后续消息会在同一个会话中继续。

```text
user: @databot what's driving Q1 revenue?  [sales.csv]
    │
    ▼
bot uploads the CSV and starts an agent session
    │
    ▼
bot streams the agent's progress back to the thread
    │
    ▼
bot posts the finished report to the thread
```

### 你将学到什么

- 通过 Slack 提及触发一次智能体运行
- 以会话串更新的形式展示智能体的进度
- 把完成的报告回发到会话串
- 通过后续回复让对话持续进行

### 前置条件

1. 运行下方的安装单元格。

2. 创建一个 [Slack 应用](https://api.slack.com/apps)：选择 **Create New App → From a manifest**，粘贴 [﷐0﷑](example_data/slack_data_bot/slack_app_manifest.yaml)，并安装到你的工作区。该清单启用了 Socket Mode（Slack 通过 WebSocket 投递事件，因此你不需要公网 URL）以及所需的权限范围。然后获取两个凭据：

   - **OAuth & Permissions** → 复制 Bot User OAuth Token（`xoxb-...`）
   - **Basic Information → App-Level Tokens** → 生成一个带有 `connections:write` 权限范围的凭据（`xapp-...`）

    在你希望机器人加入的频道中运行 `/invite @databot`。

3. 运行 [﷐0﷑](data_analyst_agent.ipynb)，它会把 `ANALYST_ENV_ID`、`ANALYST_AGENT_ID` 和 `ANALYST_AGENT_VERSION` 保存到 `.env`。

下方的设置单元格会让你输入 Slack 凭据并保存到 `.env`，这样重启后不必重新输入（也可以事先写入 `.env` 以跳过该步骤）。`.env` 已在 `.gitignore` 中——切勿把它提交到版本控制。如果你手边没有 Slack 工作区，仍然可以通读代码——每一节都说明了各自的作用——但要运行机器人就需要一个工作区。

In [ ]:
%%capture
%pip install -q "anthropic>=0.91.0" python-dotenv slack_bolt requests markdown-to-mrkdwn

In [ ]:
import io
import os
import threading
from getpass import getpass

import requests
from anthropic import Anthropic
from dotenv import load_dotenv, set_key
from markdown_to_mrkdwn import SlackMarkdownConverter
from slack_bolt import App
from slack_bolt.adapter.socket_mode import SocketModeHandler

load_dotenv(override=True)

# Prompt for Slack tokens on first run and save them to .env.
for key in ("SLACK_BOT_TOKEN", "SLACK_APP_TOKEN"):
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")
        set_key(".env", key, os.environ[key])

client = Anthropic()
app = App(token=os.environ["SLACK_BOT_TOKEN"])

for key in ("ANALYST_ENV_ID", "ANALYST_AGENT_ID", "ANALYST_AGENT_VERSION"):
    if not os.environ.get(key):
        raise RuntimeError(f"{key} not set. Run data_analyst_agent.ipynb first.")

# Set these from the IDs saved by the data analyst notebook. Reusing
# the agent and environment avoids re-provisioning on every bot restart.
ANALYST_AGENT = {
    "id": os.environ["ANALYST_AGENT_ID"],
    "version": int(os.environ["ANALYST_AGENT_VERSION"]),
}
ANALYST_ENV_ID = os.environ["ANALYST_ENV_ID"]

# thread_ts -> session_id, so follow-ups land in the same session.
# Sessions stay open for replies. In production, persist this and
# archive sessions when threads go stale.
thread_sessions: dict[str, str] = {}

mrkdwn = SlackMarkdownConverter()

## 1. 机器人被提及时启动会话

Bolt 会向每个处理器传入一个 `ack` 回调；调用它就表示已收到该事件。Slack 会对未在[三秒内](https://docs.slack.dev/apis/events-api/#responding)确认的事件进行重试，因此 `on_mention` 立即调用 `ack()`，并把耗时的工作（上传文件、创建会话、流式输出）交给后台线程中的 `start_analysis`。

每次提及都会创建一个会话，你可以在 [Console](https://platform.claude.com/) 的 **Sessions** 下打开它，查看完整的执行轨迹。

In [ ]:
@app.event("app_mention")
def on_mention(event, say, ack):
    ack()
    channel = event["channel"]
    thread_ts = event.get("thread_ts") or event["ts"]
    # Mention text arrives as "<@BOTID> question"; drop the mention prefix.
    question = event["text"].split(">", 1)[-1].strip()
    slack_file = (event.get("files") or [None])[0]

    say(text="On it. Analyzing now.", thread_ts=thread_ts)
    # Run the slow work in a background thread so this handler
    # returns within Slack's 3s limit.
    threading.Thread(target=start_analysis, args=(channel, thread_ts, question, slack_file)).start()


def start_analysis(channel: str, thread_ts: str, question: str, slack_file: dict | None) -> None:
    try:
        # If the mention had a file attached, pull it from Slack and
        # re-upload to the Anthropic Files API so the session can mount it.
        resources = []
        if slack_file:
            resp = requests.get(
                slack_file["url_private"],
                headers={"Authorization": f"Bearer {app.client.token}"},
                timeout=30,
            )
            resp.raise_for_status()
            mime = slack_file.get("mimetype", "text/csv")
            uploaded = client.beta.files.upload(
                file=(slack_file["name"], io.BytesIO(resp.content), mime)
            )
            mount = "/mnt/session/uploads/data.csv"
            resources.append({"type": "file", "file_id": uploaded.id, "mount_path": mount})
            question += f"\n\nThe data is mounted at {mount}."

        # One session per Slack thread. Store the thread coordinates in
        # metadata so anyone reading the event stream knows where to reply.
        session = client.beta.sessions.create(
            environment_id=ANALYST_ENV_ID,
            agent={"type": "agent", **ANALYST_AGENT},
            resources=resources,
            # Titles are capped at 80 chars and can't contain Unicode
            # control/format characters (Slack sometimes inserts them).
            title="".join(c for c in question if c.isprintable())[:80],
            metadata={"slack_channel": channel, "slack_thread_ts": thread_ts},
        )
        thread_sessions[thread_ts] = session.id

        # Send the question as a user.message event. The agent starts
        # working immediately; relay_stream posts its progress to the thread.
        client.beta.sessions.events.send(
            session.id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": question}]}],
        )
        relay_stream(session.id, channel, thread_ts)
    except Exception as e:
        app.client.chat_postMessage(
            channel=channel, thread_ts=thread_ts, text=f"Analysis failed: {type(e).__name__}: {e}"
        )

## 2. 把进度和结果转发到会话串

下面定义的 `relay_stream` 函数是两套 API 之间的桥梁：它从 Anthropic 的会话事件流读取内容，并发布到 Slack。它循环处理直到智能体空闲，然后发布最终摘要，并上传智能体写入的所有文件。

`files.list(scope_id=...)` 会返回会话中的每个文件——既包括我们上传的 CSV，也包括智能体写入的任何内容。我们按 `downloadable == True` 过滤，因此只有智能体生成的输出（报告、图表）会被回发到 Slack，用户自己的输入不会被回发。

In [ ]:
def relay_stream(session_id: str, channel: str, thread_ts: str) -> None:
    summary = ""
    posted_progress = False
    for ev in client.beta.sessions.events.stream(session_id):
        t = ev.type
        if t == "agent.message":
            # Keep the latest text block; it becomes the final summary.
            for b in ev.content:
                if b.type == "text" and b.text.strip():
                    summary = b.text
        elif t == "agent.tool_use" and not posted_progress:
            # Post a one-time progress update when the agent starts
            # running commands.
            app.client.chat_postMessage(
                channel=channel, thread_ts=thread_ts, text="Running analysis..."
            )
            posted_progress = True
        elif t == "session.status_idle":
            break
        elif t == "session.status_terminated":
            trace = f"https://platform.claude.com/sessions/{session_id}"
            app.client.chat_postMessage(
                channel=channel,
                thread_ts=thread_ts,
                text=f"Session terminated unexpectedly. Trace: {trace}",
            )
            return

    # Turn is done. Post the summary, then upload any generated files.
    if summary:
        text = mrkdwn.convert(summary)
        if len(text) > 3900:  # Slack text limit ~4000 chars
            text = text[:3900] + "\n_(truncated)_"
        app.client.chat_postMessage(channel=channel, thread_ts=thread_ts, text=text)
    outputs = client.beta.files.list(scope_id=session_id, betas=["managed-agents-2026-04-01"])
    for f in outputs.data:
        if not f.downloadable:
            continue
        content = client.beta.files.download(f.id).read()
        app.client.files_upload_v2(
            channel=channel, thread_ts=thread_ts, filename=f.filename, content=content
        )

## 3. 在同一会话中处理后续问题

会话串中的回复会成为现有会话中的新一轮对话——你无需再次 `@mention` 机器人。容器文件系统和对话历史会在多轮之间持续保留。

In [ ]:
def continue_session(session_id: str, channel: str, thread_ts: str, text: str) -> None:
    try:
        client.beta.sessions.events.send(
            session_id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": text}]}],
        )
        relay_stream(session_id, channel, thread_ts)
    except Exception as e:
        app.client.chat_postMessage(
            channel=channel, thread_ts=thread_ts, text=f"Analysis failed: {type(e).__name__}: {e}"
        )


@app.event("message")
def on_thread_reply(event, ack):
    ack()
    thread_ts = event.get("thread_ts")
    # Only handle human replies in a thread where we already started
    # a session. Skip edits/deletes and other message subtypes.
    if event.get("subtype"):
        return
    if not thread_ts or event.get("bot_id") or thread_ts not in thread_sessions:
        return
    threading.Thread(
        target=continue_session,
        args=(thread_sessions[thread_ts], event["channel"], thread_ts, event["text"]),
    ).start()

## 4. 运行机器人

下方的单元格会连接 Slack 并开始监听。机器人运行期间它会阻塞——完成后用 ■ 中断按钮停止。

在机器人所在的任意频道中提及它并附上 CSV。它会先发布进度，随后在会话串中给出摘要和 `report.html`：

<img src="https://raw.githubusercontent.com/anthropics/claude-cookbooks/main/managed_agents/example_data/slack_data_bot/slack_thread.png" alt="Slack thread showing the bot's analysis" width="600" />

在会话串中回复，可以就同一份数据做更深入的分析。

In [ ]:
SocketModeHandler(app, os.environ["SLACK_APP_TOKEN"]).start()

## 后续步骤

你已经把数据分析智能体封装进了 Slack 机器人：提及会启动会话，事件流把进度转发到会话串，输出会被上传，而回复会在同一对话中继续。

- 修改 [﷐0﷑](data_analyst_agent.ipynb) 中智能体的系统提示词，以改变其分析风格。重新运行该笔记本会创建一个新智能体，并把它的 ID 保存到 `.env`，供机器人读取。
- 把 `thread_sessions` 持久化到数据库，这样对话就能在机器人重启后保留。
- 把机器人搬出本笔记本：将代码复制到 `.py` 文件，并部署到任何能维持长时 WebSocket 连接的地方。